In [ ]:
import torch
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_squared_error, r2_score, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss
from torch.optim import Adam

In [ ]:
dataset = load_diabetes()

In [ ]:
x_treino, x_teste, y_treino, y_teste = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=42)

y_treino = np.expand_dims(y_treino, axis=1)
y_teste = np.expand_dims(y_teste, axis=1)

In [ ]:
print(dataset.data.shape)

In [ ]:
print(dataset.target)

In [ ]:
model = Sequential(
    Linear(10,16),
    ReLU(),
    Linear(16,16),
    ReLU(),
    Linear(16,1)
)

In [ ]:
print(model)

In [ ]:
optimizer = Adam(model.parameters(), lr=0.001)
criterion = MSELoss()
max_epochs = 100
batch = 8

In [ ]:
total_bachs = len(x_treino)//batch
last_batch = len(x_treino) % batch > 0
has_last_b = last_batch>0
print(total_bachs)
print(last_batch)

In [ ]:
def train_step_reg(x,y):
    optimizer.zero_grad()
    output = model(x)
    loss =  criterion(output, y)
    loss.backward()
    optimizer.step()

def train_model():
    for i_epochs in range(max_epochs):
        if has_last_b:
            batch_x = Tensor(x_treino[-last_batch:])
            batch_y = Tensor(y_treino[-last_batch:])
            train_step_reg(batch_x, batch_y)
            

            for i_batchs in range(total_bachs):
                        batch_x = Tensor(x_treino[(i_batchs * batch):((i_batchs + 1) * batch)])
                        batch_y = Tensor(y_treino[(i_batchs * batch):((i_batchs + 1) * batch)])
                        optimizer.zero_grad()
                        output = model(batch_x)
                        loss =  criterion(output, batch_y)
                        loss.backward()
                        optimizer.step()

def eval_model():
    with torch.no_grad():
        x_teste_tensor = Tensor(x_teste)
        output = np.asarray(model(x_teste_tensor).cpu())
        print(f"R^2: {r2_score(y_teste, output)}")
        print(f"RMSE: {root_mean_squared_error(y_teste, output)}")
        print(f"MSE: {mean_squared_error(y_teste, output)}")
        

In [ ]:
train_model()
eval_model()